# EDN-Ad v2: Fine-tune on FF++ + CelebDF → Improve cross-dataset

**Это ноутбук дообучения.** Стартует из чекпоинта первого обучения (`best_model.pt`).

## Порядок подготовки

1. ⚡ Settings → Accelerator → **GPU T4 x2**
2. 📦 Data → Add Data:
   - **`deepfake-detection-challenge`** (DFDC sample)
   - **`celeb-df-v2`** (reubensuju)
   - **`ff-c23`** (xdxd003) — FaceForensics++ C23
   - **`dfdc-faces-of-the-train-sample`** (itamargr)
   - **Output первого ноутбука** (edn-ad-training) — содержит `best_model.pt`
3. ▶️ Run All

## Что нового по сравнению с v1

- **Fine-tune из чекпоинта** (не random init) → быстрая сходимость
- **FF++ C23** — 4 метода манипуляций (DeepFakes, Face2Face, FaceSwap, NeuralTextures)
- **CelebDF-v2 часть** добавлена в train (не только eval) → прямое улучшение cross-dataset
- **Пониженный LR** (5e-5 head / 1e-6 backbone) — fine-tuning режим
- **Усиленная аугментация** — Gaussian noise, random downscale, агрессивнее JPEG

## Цель

- DFDC AUROC > 0.90 (сохранить)
- CelebDF AUROC > 0.75 (было 0.66)
- Лучшая работа на реальных данных (вебкамера, соцсети)

In [ ]:
# ─── 0. GPU + malloc ───────────────────────────────────────────────────────
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import ctypes, ctypes.util
_c_lib = ctypes.CDLL(ctypes.util.find_library('c'))
try:
    _c_lib.mallopt(-3, 65536)
    _c_lib.mallopt(-1, 131072)
    _c_lib.mallopt(-8, 2)
    print('glibc malloc tuned ✓')
except Exception as e:
    print(f'glibc tuning skipped: {e}')

def _malloc_trim():
    try: _c_lib.malloc_trim(0)
    except: pass

import subprocess, sys, importlib
def can_import(name):
    try: importlib.import_module(name); return True
    except: return False

missing = [p for p, m in [('av','av'),('timm','timm')] if not can_import(m)]
if missing:
    for pkg in missing:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg,'--no-deps'],
                       capture_output=True, timeout=180)

import numpy as np
print(f'numpy {np.__version__}')
for _, m in [('av','av'),('timm','timm'),('sklearn','sklearn'),('cv2','cv2')]:
    print(f'  {m}: {"ok" if can_import(m) else "MISSING"}')
print('Dependencies ready ✓')

In [ ]:
# ─── 1. Импорты и конфигурация ─────────────────────────────────────────────
import os, io, json, random, time, re, gc
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
import cv2, av
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

CFG = {
    # ── Модель ──
    'embed_dim':   512,
    'n_layers':    2,
    'kernel':      3,
    'dropout':     0.3,
    # ── Препроцессинг ──
    'target_frames': 16,
    'image_size':    224,
    # ── Fine-tuning ──
    'batch':      8,
    'grad_accum': 4,
    'lr':         5e-4,        # head/FPN/SE/TCN
    'lr_backbone': 1e-6,       # backbone
    'wd':         1e-4,
    'epochs':     10,
    'freeze_backbone_epochs': 1,
    'clip_grad':  5.0,
    'seed':       42,
    'use_amp':    True,
    # ── Данные ──
    'ffpp_c23':     '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23',
    'dfdc_faces':   '/kaggle/input/datasets/itamargr/dfdc-faces-of-the-train-sample',
    'celebdf_root': '/kaggle/input/datasets/reubensuju/celeb-df-v2',
    # ── Чекпоинт v1 ──
    'checkpoint_v1': '/kaggle/input/models/sashaeeeeee/best-model/other/default/1/best_model.pt',
    # ── Данные ──
    'max_per_class': 1500,
    'celebdf_train': 300,
    'celebdf_test':  200,
    'val_ratio':     0.10,
    'output_dir':    '/kaggle/working',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(CFG['seed'])
print('Config ready ✓ (fine-tune v2)')

In [ ]:
# ─── 2. Модель EDN-Ad (та же архитектура что в v1) ─────────────────────────
import timm

class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(ch, ch//r), nn.ReLU(True),
                                 nn.Linear(ch//r, ch), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b,c)).view(b,c,1,1)

class FPN(nn.Module):
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        self.lat = nn.ModuleList([nn.Conv2d(c,out_ch,1) for c in in_ch])
        self.smo = nn.ModuleList([nn.Conv2d(out_ch,out_ch,3,padding=1) for _ in in_ch])
    def forward(self, feats):
        lats = [l(f) for l,f in zip(self.lat,feats)]
        outs = [lats[-1]]
        for i in reversed(range(len(lats)-1)):
            outs.insert(0, lats[i] + F.interpolate(outs[0], size=lats[i].shape[-2:], mode='nearest'))
        return [s(o) for s,o in zip(self.smo,outs)]

class SpatialEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.bb = timm.create_model('efficientnet_b0', pretrained=True,
                                     features_only=True, out_indices=(2,3,4))
        self.fpn = FPN(self.bb.feature_info.channels())
        self.se = nn.ModuleList([SEBlock(128) for _ in range(3)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Linear(384, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        pyr = self.fpn(self.bb(x))
        pool = [self.pool(se(p)).flatten(1) for se,p in zip(self.se,pyr)]
        return self.proj(torch.cat(pool, 1))

class AdaptiveDilatedConv1d(nn.Module):
    def __init__(self, ch, k=3, bd=1, md=16):
        super().__init__()
        self.k, self.bd, self.md = k, bd, md
        self.W = nn.Parameter(torch.empty(2*ch, ch, k))
        nn.init.kaiming_uniform_(self.W, a=5**0.5)
        self.b = nn.Parameter(torch.zeros(2*ch))
    def forward(self, x, E):
        phi = 1/(1+E.mean().clamp(1e-6))
        d = int(max(1, min(self.md, round(self.bd*phi.item()))))
        y = F.conv1d(F.pad(x, ((self.k-1)*d, 0)), self.W, self.b, dilation=d)
        a, b = y.chunk(2, 1)
        return a * torch.sigmoid(b)

class TemporalBlock(nn.Module):
    def __init__(self, ch, k, bd, dr):
        super().__init__()
        self.conv = AdaptiveDilatedConv1d(ch, k, bd)
        self.ln = nn.LayerNorm(ch)
        self.drop = nn.Dropout(dr)
    def forward(self, x, E):
        y = self.conv(x, E)
        return self.drop(self.ln(y.transpose(1, 2)).transpose(1, 2)) + x

class TemporalEncoder(nn.Module):
    def __init__(self, ch=512, nl=2, k=3, dr=0.1):
        super().__init__()
        self.blks = nn.ModuleList([TemporalBlock(ch, k, 2**i, dr) for i in range(nl)])
    def forward(self, seq):
        x = seq.transpose(1, 2)
        E = F.pad((x[..., 1:] - x[..., :-1]).pow(2).mean(1, True), (1, 0))
        for blk in self.blks:
            x = blk(x, E)
        return x.mean(-1)

class DeepfakeDetector(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.spatial = SpatialEncoder(c['embed_dim'])
        self.temporal = TemporalEncoder(c['embed_dim'], c['n_layers'], c['kernel'], c['dropout'])
        self.head = nn.Sequential(nn.Dropout(c['dropout']), nn.Linear(c['embed_dim'], 1))

    def forward(self, video):
        b, t, ch, h, w = video.shape
        sp = self.spatial(video.contiguous().reshape(b*t, ch, h, w)).reshape(b, t, -1)
        agg = self.temporal(sp)
        return self.head(agg).squeeze(-1)

    def freeze_backbone(self):
        for p in self.spatial.bb.parameters(): p.requires_grad = False
        print('  Backbone FROZEN')

    def unfreeze_backbone(self):
        for p in self.spatial.bb.parameters(): p.requires_grad = True
        print('  Backbone UNFROZEN')

print('Model defined ✓')

In [ ]:
# ─── 3. Загрузка чекпоинта v1 ──────────────────────────────────────────────
# Ищем best_model.pt — мог быть в разных местах
model = DeepfakeDetector(CFG).to(device)

ckpt_candidates = [
    Path(CFG['checkpoint_v1']),
    Path('/kaggle/input/edn-ad-training/best_model.pt'),
    Path('/kaggle/working/best_model.pt'),
]
# Ищем также в любых input-папках
for p in Path('/kaggle/input').iterdir():
    if p.is_dir():
        for f in p.rglob('best_model.pt'):
            ckpt_candidates.append(f)

ckpt_path = None
for c in ckpt_candidates:
    if c.exists():
        ckpt_path = c
        break

if ckpt_path:
    print(f'Loading checkpoint: {ckpt_path}')
    ck = torch.load(ckpt_path, map_location=device)
    sd = ck.get('state_dict', ck)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:
        print(f'  Missing keys: {missing[:5]}{"..." if len(missing)>5 else ""}')
    if unexpected:
        print(f'  Unexpected keys: {unexpected[:5]}')
    print(f'  Checkpoint loaded ✓ (epoch={ck.get("epoch","?")}, metrics={ck.get("metrics",{})})')
else:
    print('⚠️  No checkpoint found — training from scratch (pretrained EfficientNet only)')
    print('   Add Output from first notebook to Data sources!')

n_p = sum(p.numel() for p in model.parameters())/1e6
print(f'Model: {n_p:.1f}M params')

In [ ]:
# ─── 4. Утилиты: face crop, video decode ───────────────────────────────────
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG')
MEAN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

_face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def _crop_face(img_rgb, margin=0.3):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    faces = _face_cascade.detectMultiScale(gray, 1.1, 4, minSize=(40,40))
    h, w = img_rgb.shape[:2]
    if len(faces) > 0:
        areas = [fw*fh for (fx,fy,fw,fh) in faces]
        fx,fy,fw,fh = faces[int(np.argmax(areas))]
        mx, my = int(fw*margin), int(fh*margin)
        x1,y1 = max(0,fx-mx), max(0,fy-my)
        x2,y2 = min(w,fx+fw+mx), min(h,fy+fh+my)
        crop = img_rgb[y1:y2, x1:x2]
        if crop.shape[0] > 10 and crop.shape[1] > 10:
            return crop
    cr = 0.65
    ch, cw = int(h*cr), int(w*cr)
    y1, x1 = (h-ch)//2, (w-cw)//2
    return img_rgb[y1:y1+ch, x1:x1+cw]

def _resize_sq(img, sz):
    h, w = img.shape[:2]
    if max(h,w) > 640:
        k = 640/max(h,w)
        img = cv2.resize(img, (int(w*k), int(h*k)), cv2.INTER_AREA)
    return cv2.resize(img, (sz, sz), cv2.INTER_LINEAR)

def _augment_frame(img):
    # Gaussian blur (p=0.3)
    if random.random() < 0.3:
        img = cv2.GaussianBlur(img, (random.choice([3,5]),)*2, 0)
    # JPEG compression (p=0.4) — усилено для generalization
    if random.random() < 0.4:
        q = random.randint(20, 75)  # более агрессивно чем v1
        _, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, q])
        img = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    # Gaussian noise (p=0.2) — новое в v2
    if random.random() < 0.2:
        noise = np.random.normal(0, random.uniform(3, 10), img.shape).astype(np.float32)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    # Random downscale→upscale (p=0.2) — имитация low-res соцсетей
    if random.random() < 0.2:
        h, w = img.shape[:2]
        scale = random.uniform(0.3, 0.7)
        small = cv2.resize(img, (int(w*scale), int(h*scale)), cv2.INTER_AREA)
        img = cv2.resize(small, (w, h), cv2.INTER_LINEAR)
    # Brightness/contrast
    if random.random() < 0.5:
        alpha = random.uniform(0.75, 1.25)
        beta = random.uniform(-20, 20)
        img = np.clip(img.astype(np.float32)*alpha + beta, 0, 255).astype(np.uint8)
    return img

print('Utilities defined ✓')

In [ ]:
# ─── 5. Сборка данных: FF++ C23 + DFDC + CelebDF ──────────────────────────

def discover_ffpp_c23(root):
    """FF++ C23 — все real + все fake (без лимита на метод)."""
    root = Path(root)
    if not root.exists():
        print(f'FF++ C23 не найден: {root}')
        return []

    print(f'Изучаю FF++ C23: {root}')
    records = []

    # Real
    orig = root / 'original'
    if orig.exists():
        mp4s = sorted(orig.glob('*.mp4'))
        for vp in mp4s:
            records.append({'path': str(vp), 'label': 0, 'dataset': 'ffpp_c23',
                             'manipulation': 'original', 'source_type': 'video'})
        print(f'  Real [original]: {len(mp4s)} videos')

    # Fake — все методы, все видео
    fake_methods = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures',
                    'FaceShifter', 'DeepFakeDetection']
    for method in fake_methods:
        method_dir = root / method
        if method_dir.exists():
            mp4s = sorted(method_dir.glob('*.mp4'))
            for vp in mp4s:
                records.append({'path': str(vp), 'label': 1, 'dataset': 'ffpp_c23',
                                 'manipulation': method, 'source_type': 'video'})
            print(f'  Fake [{method}]: {len(mp4s)} videos')

    r = sum(1 for x in records if x['label']==0)
    f = sum(1 for x in records if x['label']==1)
    print(f'FF++ C23 ИТОГО: real={r}, fake={f}')
    return records


def discover_dfdc_faces(root, max_per_class=1500):
    """DFDC faces — PNG-кропы лиц, сгруппированные по видео-ID."""
    root = Path(root)
    if not root.exists():
        print(f'DFDC faces не найден: {root}')
        return []
    print(f'Изучаю DFDC faces: {root}')

    records = []
    for split in ['train', 'validation']:
        for sub, lbl in [('real', 0), ('fake', 1)]:
            d = root / split / sub
            if not d.exists():
                continue
            files = sorted(d.glob('*.png'))
            if not files:
                continue
            groups = {}
            for f in files:
                vid_id = f.stem.split('_')[0]
                groups.setdefault(vid_id, []).append(str(f))
            for vid_id, flist in groups.items():
                if len(flist) >= 2:
                    records.append({
                        'path': str(d), 'files': sorted(flist),
                        'label': lbl, 'dataset': 'dfdc_faces',
                        'manipulation': 'dfdc', 'source_type': 'files',
                    })
            print(f'  {split}/{sub}: {len(files)} files → {len(groups)} video-groups')

    real = [r for r in records if r['label']==0]
    fake = [r for r in records if r['label']==1]
    random.shuffle(real); random.shuffle(fake)
    real, fake = real[:max_per_class], fake[:max_per_class]
    print(f'DFDC faces: real={len(real)}, fake={len(fake)}')
    return real + fake


def discover_celebdf(root, n_train=300, n_test=200):
    """CelebDF-v2 — часть в train, часть в eval."""
    root = Path(root)
    if not root.exists():
        print(f'CelebDF не найден: {root}')
        return [], []

    print(f'Изучаю CelebDF: {root}')
    all_records = []
    for folder, lbl in [('Celeb-real', 0), ('YouTube-real', 0), ('Celeb-synthesis', 1)]:
        fp = root / folder
        if fp.exists():
            mp4s = sorted(fp.glob('*.mp4'))
            for vp in mp4s:
                all_records.append({'path': str(vp), 'label': lbl, 'dataset': 'celebdf',
                                     'manipulation': 'celebdf', 'source_type': 'video'})
            print(f'  {folder}: {len(mp4s)} videos')

    real = [r for r in all_records if r['label']==0]
    fake = [r for r in all_records if r['label']==1]
    random.shuffle(real); random.shuffle(fake)

    nt = min(n_train//2, len(real)//2, len(fake)//2)
    ne = min(n_test//2, len(real)//2 - nt, len(fake)//2 - nt)

    train_recs = real[:nt] + fake[:nt]
    test_recs  = real[nt:nt+ne] + fake[nt:nt+ne]
    random.shuffle(train_recs); random.shuffle(test_recs)

    print(f'CelebDF: total={len(all_records)} (real={len(real)}, fake={len(fake)})')
    print(f'  → train: {len(train_recs)} | test: {len(test_recs)}')
    return train_recs, test_recs


# ── Собираем всё ──
print('='*60)
print('=== FF++ C23 ===')
ffpp_records = discover_ffpp_c23(CFG['ffpp_c23'])

print('\n=== DFDC Faces ===')
dfdc_records = discover_dfdc_faces(CFG['dfdc_faces'], CFG['max_per_class'])

print('\n=== CelebDF-v2 ===')
celebdf_train, celebdf_test = discover_celebdf(
    CFG['celebdf_root'], CFG['celebdf_train'], CFG['celebdf_test'])

# Объединяем
all_train = ffpp_records + dfdc_records + celebdf_train
random.shuffle(all_train)

n_val = max(20, int(len(all_train) * CFG['val_ratio']))
train_recs = all_train[n_val:]
val_recs   = all_train[:n_val]

r_cnt = sum(1 for x in train_recs if x['label']==0)
f_cnt = sum(1 for x in train_recs if x['label']==1)
ratio = f_cnt / max(r_cnt, 1)
print(f'\n{"="*60}')
print(f'ИТОГО: train={len(train_recs)} (real={r_cnt}, fake={f_cnt}, ratio={ratio:.2f})')
print(f'       val={len(val_recs)}, celebdf_test={len(celebdf_test)}')
print(f'{"="*60}')

In [ ]:
# ─── 6. Precache видео → JPG + face crop ──────────────────────────────────
# 2 воркера + батчи по 50 + gc между батчами — баланс скорости и памяти.

CACHE = Path('/kaggle/working/cache_v2')
CACHE.mkdir(parents=True, exist_ok=True)

def _decode_one(video_path, out_dir, T=16, sz=224):
    out_dir = Path(out_dir)
    if out_dir.exists() and len(list(out_dir.glob('f_*.jpg'))) >= T:
        return True
    out_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    try:
        with av.open(str(video_path)) as c:
            st = c.streams.video[0]
            total = st.frames or 0
            if total <= 0: idx = set()
            elif total <= T: idx = set(range(total))
            else: idx = {int(total/T*k) for k in range(T)}
            i = done = 0
            for pkt in c.demux(st):
                for fr in pkt.decode():
                    if not idx or i in idx:
                        img = fr.to_ndarray(format='rgb24')
                        h, w = img.shape[:2]
                        if max(h,w) > 640:
                            k = 640/max(h,w)
                            img = cv2.resize(img, (int(w*k), int(h*k)), cv2.INTER_AREA)
                        img = _crop_face(img)
                        frames.append(cv2.resize(img, (sz,sz), cv2.INTER_LINEAR))
                    i += 1
                    if len(frames) == T: done = 1; break
                if done: break
    except:
        return False
    if not frames:
        return False
    while len(frames) < T:
        frames.append(frames[-1].copy())
    for k, im in enumerate(frames):
        cv2.imwrite(str(out_dir/f'f_{k:03d}.jpg'),
                    cv2.cvtColor(im, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 92])
    return True

def precache(records, tag='', batch_size=50, max_workers=2):
    """Декодируем видео батчами по 50, 2 воркера, gc между батчами."""
    vids = [r for r in records if r.get('source_type') == 'video']
    if not vids:
        return records
    jobs = []
    for r in vids:
        vid_id = Path(r['path']).stem
        ds = r.get('dataset', 'x')
        dst = CACHE / ds / r.get('manipulation', 'unk') / vid_id
        jobs.append((r, dst))
    n_batches = (len(jobs) - 1) // batch_size + 1
    print(f'{tag}: decoding {len(jobs)} mp4 → JPG ({n_batches} batches x {batch_size}, {max_workers} workers)...')
    ok = 0
    for bi in range(0, len(jobs), batch_size):
        batch = jobs[bi:bi + batch_size]
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = {
                ex.submit(_decode_one, r['path'], dst, CFG['target_frames'], CFG['image_size']): (r, dst)
                for r, dst in batch
            }
            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=f'{tag} batch {bi//batch_size+1}/{n_batches}', leave=False):
                r, dst = futures[fut]
                try:
                    if fut.result():
                        r['path'] = str(dst)
                        r['source_type'] = 'folder'
                        ok += 1
                except:
                    pass
        gc.collect()
        _malloc_trim()
    print(f'{tag}: done {ok}/{len(jobs)}')
    return records

train_recs = precache(train_recs, 'train')
val_recs   = precache(val_recs, 'val')
gc.collect(); _malloc_trim()
cache_files = list(CACHE.rglob('*.jpg'))
print(f'Cache: {len(cache_files)} JPG files, {sum(f.stat().st_size for f in cache_files)/1e6:.0f} MB')

In [ ]:
# ─── 7. Dataset с ленивой загрузкой с диска (не в RAM — экономим 50GB) ─────
# Вместо preload в RAM — читаем фреймы с диска при каждом __getitem__.
# Кеш уже на SSD, поэтому чтение быстрое.

def _load_frames_u8(rec, T, sz):
    """Загрузить T фреймов из record (folder с JPG или files с PNG)."""
    src = rec.get('source_type')
    if src == 'files':
        files = sorted(rec['files'])
    elif src == 'folder':
        p = Path(rec['path'])
        files = []
        for ext in IMG_EXTS:
            files.extend(p.glob(f'*{ext}'))
        files = sorted(str(f) for f in files)
    else:
        return None
    if not files:
        return None
    n = len(files)
    idx = list(range(n)) if n <= T else [int(n / T * k) for k in range(T)]
    frames = []
    for i in idx:
        img = cv2.imread(str(files[i]), cv2.IMREAD_COLOR)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        frames.append(_resize_sq(img, sz))
    while len(frames) < T:
        frames.append(frames[-1].copy() if frames else np.zeros((sz, sz, 3), np.uint8))
    return np.stack(frames[:T]).astype(np.uint8)

print('Lazy loading utilities defined ✓')

In [ ]:
# ─── 8. Dataset, Loss, Metrics ─────────────────────────────────────────────

bce_loss = nn.BCEWithLogitsLoss()

def compute_metrics(labels, probs):
    lb, pb = np.array(labels), np.array(probs)
    if len(set(lb)) < 2:
        return {'auroc': 0., 'f1': 0., 'eer': 0.}
    auroc = float(roc_auc_score(lb, pb))
    f1 = float(f1_score(lb, (pb >= 0.5).astype(int), zero_division=0))
    fpr, tpr, _ = roc_curve(lb, pb)
    fnr = 1 - tpr
    eer = float((fpr + fnr)[np.argmin(np.abs(fnr - fpr))] / 2)
    return {'auroc': auroc, 'f1': f1, 'eer': eer}


class LazyDS(Dataset):
    """Ленивый Dataset — читает фреймы с диска по мере необходимости."""
    def __init__(self, records, T=16, sz=224, augment=False):
        self.recs = records
        self.T, self.sz = T, sz
        self.aug = augment

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, idx):
        rec = self.recs[idx]
        arr = _load_frames_u8(rec, self.T, self.sz)
        if arr is None:
            arr = np.zeros((self.T, self.sz, self.sz, 3), dtype=np.uint8)
        if self.aug:
            for i in range(len(arr)):
                arr[i] = _augment_frame(arr[i])
        vid = torch.from_numpy(arr).float().permute(0, 3, 1, 2).div_(255.0)
        vid.sub_(MEAN).div_(STD)
        if self.aug and random.random() < 0.5:
            vid = vid.flip(-1)
        return {'video': vid, 'label': int(rec['label'])}


# Для CelebDF eval — декодируем видео на лету
def load_video_fc(path, T=16, sz=224):
    frames = []
    try:
        with av.open(str(path)) as c:
            st = c.streams.video[0]
            total = st.frames or 0
            if total <= 0: idx = set()
            elif total <= T: idx = set(range(total))
            else: idx = {int(total / T * k) for k in range(T)}
            i = done = 0
            for pkt in c.demux(st):
                for fr in pkt.decode():
                    if not idx or i in idx:
                        img = fr.to_ndarray(format='rgb24')
                        h, w = img.shape[:2]
                        if max(h, w) > 640:
                            k = 640 / max(h, w)
                            img = cv2.resize(img, (int(w * k), int(h * k)), cv2.INTER_AREA)
                        img = _crop_face(img)
                        frames.append(cv2.resize(img, (sz, sz), cv2.INTER_LINEAR))
                    i += 1
                    if len(frames) == T: done = 1; break
                if done: break
    except: pass
    while len(frames) < T:
        frames.append(frames[-1].copy() if frames else np.zeros((sz, sz, 3), np.uint8))
    arr = torch.from_numpy(np.stack(frames[:T])).float() / 255.
    arr = arr.permute(0, 3, 1, 2)
    return (arr - MEAN) / STD


class VideoDS(Dataset):
    def __init__(self, records, T=16, sz=224):
        self.recs, self.T, self.sz = records, T, sz
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, idx):
        r = self.recs[idx]
        vid = load_video_fc(r['path'], self.T, self.sz)
        return {'video': vid, 'label': int(r['label'])}

print('Dataset & loss defined ✓')

In [ ]:
# ─── 9. Training loop ──────────────────────────────────────────────────────
def run_epoch(model, loader, opt, scaler, accum, dev, train=True, desc=''):
    model.train() if train else model.eval()
    total_loss, n = 0., 0
    all_p, all_l = [], []
    if train and opt: opt.zero_grad(set_to_none=True)
    ctx = torch.enable_grad() if train else torch.no_grad()
    pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True, mininterval=1.0)
    with ctx:
        for step, batch in enumerate(pbar):
            v = batch['video'].to(dev, non_blocking=True)
            l = batch['label'].to(dev, non_blocking=True).float()
            with torch.amp.autocast('cuda', enabled=CFG['use_amp']):
                logits = model(v)
                loss = bce_loss(logits, l)
                if train: loss = loss / accum
            if train:
                scaler.scale(loss).backward()
                total_loss += loss.item() * accum
                if (step+1) % accum == 0:
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(), CFG['clip_grad'])
                    scaler.step(opt); scaler.update()
                    opt.zero_grad(set_to_none=True)
                pbar.set_postfix(loss=f'{loss.item()*accum:.3f}')
            else:
                total_loss += loss.item()
                all_p += torch.sigmoid(logits).cpu().tolist()
                all_l += l.cpu().int().tolist()
            n += 1
            if step % 50 == 49: gc.collect(); _malloc_trim()
        if train and opt and n > 0 and n % accum != 0:
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG['clip_grad'])
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
    gc.collect(); torch.cuda.empty_cache(); _malloc_trim()
    avg_loss = total_loss / max(n, 1)
    if train: return avg_loss
    m = compute_metrics(all_l, all_p)
    m['loss'] = avg_loss
    return m

print('Training loop defined ✓')

In [ ]:
# ─── 10. Init optimizer, scheduler ─────────────────────────────────────────
torch.backends.cudnn.benchmark = False

train_ds = LazyDS(train_recs, T=CFG['target_frames'], sz=CFG['image_size'], augment=True)
val_ds   = LazyDS(val_recs,   T=CFG['target_frames'], sz=CFG['image_size'], augment=False)
train_ld = DataLoader(train_ds, batch_size=CFG['batch'], shuffle=True, num_workers=2,
                      drop_last=True, pin_memory=True, persistent_workers=True)
val_ld   = DataLoader(val_ds, batch_size=CFG['batch']*2, shuffle=False, num_workers=2,
                      pin_memory=True, persistent_workers=True)

model.freeze_backbone()

bb_params = list(model.spatial.bb.parameters())
bb_ids = {id(p) for p in bb_params}
other_params = [p for p in model.parameters() if id(p) not in bb_ids]

opt = AdamW([
    {'params': bb_params,     'lr': CFG['lr_backbone']},  # 5e-7
    {'params': other_params,  'lr': CFG['lr']},            # 2e-4
], weight_decay=CFG['wd'])

sched = CosineAnnealingLR(opt, T_max=CFG['epochs'])
scaler = torch.amp.GradScaler(enabled=CFG['use_amp'])

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
print(f'Trainable: {n_trainable:.1f}M (backbone frozen)')
print(f'Train: {len(train_ds)} samples, {len(train_ld)} batches')
print(f'Val: {len(val_ds)} samples, {len(val_ld)} batches')
print(f'LR: backbone={CFG["lr_backbone"]}, head={CFG["lr"]}')
print(f'Epochs: {CFG["epochs"]}, freeze: {CFG["freeze_backbone_epochs"]}')

In [ ]:
# ─── 11. Training ──────────────────────────────────────────────────────────
import pandas as pd

best_auroc = 0.
out = Path(CFG['output_dir'])
history = []
freeze_ep = CFG['freeze_backbone_epochs']

print(f'Fine-tuning {CFG["epochs"]} epochs')
print(f'Phase 1 (ep 0-{freeze_ep-1}): backbone FROZEN')
print(f'Phase 2 (ep {freeze_ep}+): backbone UNFROZEN with lr={CFG["lr_backbone"]}')
print('='*70)

for epoch in tqdm(range(CFG['epochs']), desc='Epochs'):
    gc.collect(); torch.cuda.empty_cache()

    if epoch == freeze_ep:
        model.unfreeze_backbone()

    t0 = time.time()
    tr_loss = run_epoch(model, train_ld, opt, scaler, CFG['grad_accum'],
                        device, train=True, desc=f'train ep{epoch}')
    val_m = run_epoch(model, val_ld, None, None, 1,
                      device, train=False, desc=f'val ep{epoch}')
    sched.step()
    dt = time.time() - t0

    row = dict(epoch=epoch, train_loss=round(tr_loss,4),
               val_loss=round(val_m['loss'],4), auroc=round(val_m['auroc'],4),
               f1=round(val_m['f1'],4), eer=round(val_m['eer'],4), time_s=round(dt,1))
    history.append(row)

    frozen = ' [FROZEN]' if epoch < freeze_ep else ''
    print(f"Ep{epoch:02d} | tr={tr_loss:.4f} | val={val_m['loss']:.4f} | "
          f"AUROC={val_m['auroc']:.4f} | F1={val_m['f1']:.4f}{frozen} | {dt:.0f}s")

    # Save checkpoint
    torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                'sched': sched.state_dict(), 'best_auroc': best_auroc,
                'history': history, 'cfg': CFG}, out/'last_model.pt')

    if val_m['auroc'] > best_auroc:
        best_auroc = val_m['auroc']
        torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                    'metrics': val_m, 'cfg': CFG}, out/'best_model.pt')
        print(f'  ★ New best AUROC {best_auroc:.4f}')

print('='*70)
print(f'Training done. Best AUROC: {best_auroc:.4f}')
df_hist = pd.DataFrame(history)
df_hist.to_csv(out/'training_history.csv', index=False)

In [ ]:
# ─── 12. Cross-dataset evaluation ──────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

# Load best model
ckpt = torch.load(out/'best_model.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

# DFDC val (in-domain)
dfdc_m = run_epoch(model, val_ld, None, None, 1, device, train=False, desc='DFDC val')
print(f"DFDC val: AUROC={dfdc_m['auroc']:.4f} | F1={dfdc_m['f1']:.4f} | EER={dfdc_m['eer']:.4f}")

# CelebDF-v2 (cross-dataset)
if celebdf_test:
    ds_cd = VideoDS(celebdf_test, T=CFG['target_frames'], sz=CFG['image_size'])
    ld_cd = DataLoader(ds_cd, batch_size=CFG['batch']*2, num_workers=0)
    cd_m = run_epoch(model, ld_cd, None, None, 1, device, train=False, desc='CelebDF-v2')
    print(f"CelebDF-v2: AUROC={cd_m['auroc']:.4f} | F1={cd_m['f1']:.4f} | EER={cd_m['eer']:.4f}")
else:
    cd_m = {'auroc': None, 'f1': None, 'eer': None}

# ── Results table ──
print('\n' + '='*60)
print('  ИТОГОВЫЕ РЕЗУЛЬТАТЫ (EDN-Ad v2 Fine-tuned)')
print('='*60)
results = pd.DataFrame([
    {'Dataset': 'DFDC (val)', 'AUROC': round(dfdc_m['auroc'],4),
     'F1': round(dfdc_m['f1'],4), 'EER': round(dfdc_m['eer'],4)},
    {'Dataset': 'CelebDF-v2 (cross)', 'AUROC': round(cd_m['auroc'],4) if cd_m['auroc'] else 'N/A',
     'F1': round(cd_m['f1'],4) if cd_m['f1'] else 'N/A',
     'EER': round(cd_m['eer'],4) if cd_m['eer'] else 'N/A'},
])
print(results.to_string(index=False))
print('='*60)
results.to_csv(out/'final_results.csv', index=False)

# ── Plots ──
df_h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('EDN-Ad v2 Fine-tuning', fontsize=13)

axes[0].plot(df_h['epoch'], df_h['train_loss'], label='Train')
axes[0].plot(df_h['epoch'], df_h['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(df_h['epoch'], df_h['auroc'], 'g-o', label='Val AUROC')
if cd_m['auroc']:
    axes[1].axhline(cd_m['auroc'], color='orange', ls='--', label=f'CelebDF {cd_m["auroc"]:.3f}')
axes[1].set_title('AUROC'); axes[1].set_ylim(0,1); axes[1].legend(); axes[1].grid(True)

axes[2].plot(df_h['epoch'], df_h['f1'], 'b-', label='F1')
axes[2].plot(df_h['epoch'], df_h['eer'], 'r-', label='EER')
axes[2].set_title('F1 & EER'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig(out/'results_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nOutput files:')
print('  best_model.pt        — fine-tuned weights')
print('  final_results.csv    — results table')
print('  results_v2.png       — training plots')
print('  training_history.csv — per-epoch history')